# Etapa 2 — Engenharia de características

Transforma a base da etapa 1 (uma linha por equipe em cada partida) em **uma linha por
partida**, com variáveis que descrevem as duas equipes usando **apenas jogos anteriores** à
partida a prever, e define os folds da validação Walk-Forward.

| | Arquivo | Conteúdo |
|---|---|---|
| Entrada | `dados/processados/partidas_times.parquet` | saída da etapa 1 |
| Saída | `dados/processados/features_jogos.parquet` | partidas × variáveis preditoras |
| Saída | `dados/processados/folds_walkforward.csv` | composição das 18 folds de validação |
| Saída | `dados/processados/metadados_features.json` | parâmetros e lista final de variáveis |

**Regras contra o vazamento de informação** (*data leakage*: o modelo receber, durante o
treino, algo que não estaria disponível no momento da previsão):

1. Toda estatística é deslocada uma posição dentro do grupo (equipe, temporada): a linha da
   partida G só enxerga as partidas anteriores a G.
2. As janelas nunca atravessam a virada de temporada, porque entre uma temporada e outra o
   elenco muda.
3. Nada aqui é ajustado aos dados (nenhuma padronização ou imputação). Tudo que aprende com os
   dados fica na etapa 3, dentro de cada fold da validação.

## 1. Parâmetros

- **Janelas**: as 5 e as 10 partidas anteriores (`j5`, `j10`) e o acumulado da temporada até a
  véspera (`tmp`). As curtas captam a fase recente; a da temporada, o nível estável da equipe.
- **Perfil de pontuação só em `j10` e `tmp`**: estilo de jogo é traço estável da equipe, e cinco
  partidas dariam uma estimativa ruidosa.
- **Aquecimento**: a partida só entra se as duas equipes já tiverem disputado 10 jogos na
  temporada, para que as janelas estejam completas.
- **Ausentes**: variável com mais de 20% de valores ausentes em alguma temporada é descartada.
  Preencher uma temporada inteira inventaria dado e distorceria a comparação entre as eras.
- **Descanso**: dias sem jogo, limitados a 5 (a partir daí o efeito não muda).
- **Temporada anômala**: 2012, disputada com calendário reduzido por causa do *lockout*. É
  mantida e marcada.

In [2]:
from datetime import datetime
from pathlib import Path
import json

import numpy as np
import pandas as pd

pd.set_option("display.width", 200)

DIR_DADOS = Path("dados/processados")

JANELAS = [5, 10]
JANELAS_PERFIL = ["j10", "tmp"]
MIN_JOGOS_AQUECIMENTO = 10
LIMITE_AUSENTES_TEMPORADA = 0.20
DESCANSO_MAXIMO = 5
TEMPORADAS_ANOMALAS = {2012}

CHAVE = ["team_id", "temporada"]       # toda janela é calculada dentro deste grupo

# Estatísticas somadas nas janelas (da equipe e, com prefixo opp_, do adversário)
BRUTAS = ["pts", "fgm", "fga", "fg3m", "fg3a", "ftm", "fta",
          "orb", "drb", "ast", "stl", "blk", "tov", "pf"]
PERFIL = ["pts_garrafao", "pts_contra_ataque", "pts_segunda_chance",
          "pts_erro_adv", "pts_banco", "num_meia_distancia", "num_3p_assistidos"]

base = pd.read_parquet(DIR_DADOS / "partidas_times.parquet")
print(f"entrada: {len(base):,} linhas | {base['game_id'].nunique():,} partidas")

entrada: 48,700 linhas | 24,350 partidas


## 2. Medidas avançadas calculadas a partir de somas

Cada medida é calculada sobre as **somas** da janela (soma dos numeradores ÷ soma dos
denominadores), nunca como média de cada jogo. Assim cada partida pesa pelo seu
volume, e as taxas por posse de bola já neutralizam o efeito das prorrogações.

Posses de bola (descrita na literatura): `POSS = FGA + 0,44·FTA − ORB + TOV`, e o análogo para o
adversário.

| Grupo | Medida |
|---|---|
| Eficiência por 100 posses | `ortg` = 100·PTS/POSS; `drtg` = 100·PTS do adversário/POSS do adversário; `net_rtg` = `ortg` − `drtg` |
| Aproveitamento | `efg` = (FGM + 0,5·3PM)/FGA; `ts` = PTS/(2·(FGA + 0,44·FTA)); `fg3_pct`; `ft_pct` |
| Perfil de arremesso | `fg3ar` = 3PA/FGA (volume de três); `ftr` = FTA/FGA |
| Posse e rebote | `tov_pct` = TOV/POSS; `orb_pct` = ORB/(ORB + DRB do adversário); `drb_pct` = DRB/(DRB + ORB do adversário) |
| Ações por 100 posses | assistências, roubos, bloqueios e faltas |
| Desempenho concedido | `opp_efg`, `opp_fg3ar`, `opp_tov_pct`: o que a equipe permite ao adversário |
| Ritmo | `pace` = posses médias por jogo das duas equipes |
| Perfil de pontuação | fração dos pontos em meia distância e após erro do adversário; fração das cestas de três assistidas; e (descartadas na seção 6) garrafão, contra-ataque, segunda chance e banco |
| Resultado | `win_pct` = proporção de vitórias na janela |

In [3]:
def div(num, den):
    # Divisão protegida: denominador nulo, zero ou negativo vira NaN.
    num = np.asarray(num, dtype="float64")
    den = np.asarray(den, dtype="float64")
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.where(den > 0, num / den, np.nan)


def medidas_avancadas(s):
    # s: somas da janela (colunas BRUTAS, opp_BRUTAS e n_jogos)
    poss = s["fga"] + 0.44 * s["fta"] - s["orb"] + s["tov"]
    poss_opp = s["opp_fga"] + 0.44 * s["opp_fta"] - s["opp_orb"] + s["opp_tov"]
    m = pd.DataFrame({
        "ortg": div(100 * s["pts"], poss),
        "drtg": div(100 * s["opp_pts"], poss_opp),
        "efg": div(s["fgm"] + 0.5 * s["fg3m"], s["fga"]),
        "ts": div(s["pts"], 2 * (s["fga"] + 0.44 * s["fta"])),
        "fg3_pct": div(s["fg3m"], s["fg3a"]),
        "ft_pct": div(s["ftm"], s["fta"]),
        "fg3ar": div(s["fg3a"], s["fga"]),
        "ftr": div(s["fta"], s["fga"]),
        "tov_pct": div(s["tov"], poss),
        "orb_pct": div(s["orb"], s["orb"] + s["opp_drb"]),
        "drb_pct": div(s["drb"], s["drb"] + s["opp_orb"]),
        "ast_per100": div(100 * s["ast"], poss),
        "stl_per100": div(100 * s["stl"], poss),
        "blk_per100": div(100 * s["blk"], poss),
        "pf_per100": div(100 * s["pf"], poss),
        "opp_efg": div(s["opp_fgm"] + 0.5 * s["opp_fg3m"], s["opp_fga"]),
        "opp_fg3ar": div(s["opp_fg3a"], s["opp_fga"]),
        "opp_tov_pct": div(s["opp_tov"], poss_opp),
        "pace": (poss + poss_opp) / 2 / s["n_jogos"].where(s["n_jogos"] > 0),
    }, index=s.index)
    m["net_rtg"] = m["ortg"] - m["drtg"]
    return m


def medidas_perfil(s):
    # Frações do total de pontos: comparáveis entre eras de ritmo diferente.
    return pd.DataFrame({
        "pct_garrafao": div(s["pts_garrafao"], s["pts"]),
        "pct_contra_ataque": div(s["pts_contra_ataque"], s["pts"]),
        "pct_segunda_chance": div(s["pts_segunda_chance"], s["pts"]),
        "pct_erro_adv": div(s["pts_erro_adv"], s["pts"]),
        "pct_banco": div(s["pts_banco"], s["pts"]),
        "pct_meia_distancia": div(s["num_meia_distancia"], s["pts"]),
        "pct_3p_assistidos": div(s["num_3p_assistidos"], s["fg3m"]),
        "opp_pct_garrafao": div(s["opp_pts_garrafao"], s["opp_pts"]),
        "opp_pct_contra_ataque": div(s["opp_pts_contra_ataque"], s["opp_pts"]),
    }, index=s.index)


uma_linha = base.head(1).assign(n_jogos=1.0)
print("medidas de cada janela:", list(medidas_avancadas(uma_linha).columns))
print("medidas de perfil (só j10 e tmp):", list(medidas_perfil(uma_linha).columns))

medidas de cada janela: ['ortg', 'drtg', 'efg', 'ts', 'fg3_pct', 'ft_pct', 'fg3ar', 'ftr', 'tov_pct', 'orb_pct', 'drb_pct', 'ast_per100', 'stl_per100', 'blk_per100', 'pf_per100', 'opp_efg', 'opp_fg3ar', 'opp_tov_pct', 'pace', 'net_rtg']
medidas de perfil (só j10 e tmp): ['pct_garrafao', 'pct_contra_ataque', 'pct_segunda_chance', 'pct_erro_adv', 'pct_banco', 'pct_meia_distancia', 'pct_3p_assistidos', 'opp_pct_garrafao', 'opp_pct_contra_ataque']


## 3. Janelas por equipe

`somas_janela` soma cada estatística nas partidas **anteriores** da mesma equipe na mesma
temporada: primeiro desloca a série uma posição (`shift(1)`, que tira a partida atual) e depois
soma as últimas *k* partidas (`rolling`) ou todas desde o início da temporada (`expanding`).
Enquanto a equipe não tiver *k* jogos anteriores, a janela de *k* fica vazia.

A soma de pontos da janela de 5 jogos é guardada à parte (`_check_soma_pts`) só para a
verificação da seção 4.

In [4]:
def somas_janela(d, cols, w=None):
    # w=None: acumulado da temporada até a véspera; w=k: últimos k jogos.
    passado = d.groupby(CHAVE, sort=False)[cols].shift(1)
    grupos = passado.groupby([d[c] for c in CHAVE], sort=False)
    res = (grupos.expanding(min_periods=1).sum() if w is None
           else grupos.rolling(w, min_periods=w).sum())
    return res.reset_index(level=list(range(len(CHAVE))), drop=True).sort_index()


d = (base.sort_values(["team_id", "temporada", "data", "game_id"])
         .reset_index(drop=True))
d["n_jogos"] = 1.0
d["vitoria_f"] = d["vitoria"].astype("float64")

cols_soma = (BRUTAS + [f"opp_{c}" for c in BRUTAS]
             + PERFIL + [f"opp_{c}" for c in PERFIL] + ["n_jogos", "vitoria_f"])
blocos = [d]
for nome, w in [(f"j{w}", w) for w in JANELAS] + [("tmp", None)]:
    s = somas_janela(d, cols_soma, w)
    met = medidas_avancadas(s)
    if nome in JANELAS_PERFIL:
        met = pd.concat([met, medidas_perfil(s)], axis=1)
    met["win_pct"] = div(s["vitoria_f"], s["n_jogos"])
    blocos.append(met.add_prefix(f"{nome}_"))
    if nome == f"j{JANELAS[0]}":
        blocos.append(s[["pts"]].rename(columns={"pts": "_check_soma_pts"}))
    print(f"janela {nome}: {met.shape[1]} medidas")
d = pd.concat(blocos, axis=1)
d["jogos_anteriores"] = d.groupby(CHAVE, sort=False).cumcount()

janela j5: 21 medidas
janela j10: 30 medidas
janela tmp: 30 medidas


### Contexto de calendário

- `rest_days`: dias sem jogo antes da partida (0 = jogou na véspera), limitado a 5; a primeira
  partida da temporada recebe o máximo.
- `back_to_back`: 1 quando a equipe jogou no dia anterior.
- `jogos_7d`: partidas disputadas nos 7 dias anteriores, sem contar o dia do jogo.

In [4]:
dia = d["data"].dt.normalize()
dia_anterior = dia.groupby([d[c] for c in CHAVE], sort=False).shift(1)
d["rest_days"] = (((dia - dia_anterior).dt.days - 1)
                  .fillna(DESCANSO_MAXIMO).clip(0, DESCANSO_MAXIMO))
d["back_to_back"] = (d["rest_days"] == 0).astype("int8")

aux = d[CHAVE].assign(dia=dia, um=1.0)
carga = (aux.set_index("dia").groupby(CHAVE, sort=False)["um"]
            .rolling("7D", closed="left").sum())
d["jogos_7d"] = carga.fillna(0).to_numpy()   # mesma ordem de d: os grupos são contíguos

d = d.drop(columns=["n_jogos", "vitoria_f"])
print(d[["rest_days", "back_to_back", "jogos_7d"]].describe().round(2).to_string())

       rest_days  back_to_back  jogos_7d
count   48700.00      48700.00  48700.00
mean        1.12          0.22      3.10
std         0.97          0.41      0.86
min         0.00          0.00      0.00
25%         1.00          0.00      3.00
50%         1.00          0.00      3.00
75%         1.00          0.00      4.00
max         5.00          1.00      5.00


## 4. Verificação de vazamento

Recalcula de forma independente, com um laço explícito, a soma de pontos das 5 partidas
anteriores em 500 linhas sorteadas e compara com o valor da janela vetorizada. Se a janela
incluísse a própria partida, ou partidas de outra temporada, os dois valores divergiriam.

In [5]:
def verificar_vazamento(d, amostra=500, semente=42):
    w = JANELAS[0]
    rng = np.random.default_rng(semente)
    candidatas = d.index[d["jogos_anteriores"] >= w]
    idx = rng.choice(candidatas, size=min(amostra, len(candidatas)), replace=False)
    erros = 0
    for i in idx:
        lin = d.loc[i]
        anteriores = d[(d["team_id"] == lin["team_id"]) & (d["temporada"] == lin["temporada"])
                       & (d["data"] < lin["data"])].sort_values("data")
        if not np.isclose(anteriores["pts"].tail(w).sum(), lin["_check_soma_pts"]):
            erros += 1
    print(f"{len(idx)} linhas conferidas: {erros} divergências")
    assert erros == 0, "A janela inclui dados indevidos."


verificar_vazamento(d)

500 linhas conferidas: 0 divergências


## 5. Uma linha por partida: diferenciais com sinal

Cada variável vira um **diferencial com sinal**: valor do mandante − valor do visitante.
Positivo indica vantagem do mandante naquela medida; o valor absoluto apagaria qual das duas
equipes é a melhor. Estratégia descrita na literatura.

As variáveis de calendário entram também em **nível** (`casa_` e `fora_`).

Não há variável de mando de quadra: com o mandante sempre na mesma posição, ela valeria 1 em
todas as partidas. A vantagem de jogar em casa aparece como a própria taxa de vitórias do
mandante, que é a referência (classificador da moda) da etapa 3.

In [6]:
def colunas_features(jogos):
    return [c for c in jogos.columns
            if c.startswith(("dif_", "casa_", "fora_")) or c == "temporada_anomala"]


prefixos = tuple(f"j{w}_" for w in JANELAS) + ("tmp_",)
feats_janela = [c for c in d.columns if c.startswith(prefixos)]
contexto = ["rest_days", "back_to_back", "jogos_7d"]

casa = d[d["mandante"] == 1].set_index("game_id")
fora = d[d["mandante"] == 0].set_index("game_id").reindex(casa.index)

jogos = casa[["data", "temporada", "era"]].copy()
jogos["alvo"] = casa["vitoria_mandante"].astype("int8")
jogos["home_team_id"] = casa["team_id"]
jogos["away_team_id"] = fora["team_id"]
jogos["min_jogos_anteriores"] = np.minimum(casa["jogos_anteriores"], fora["jogos_anteriores"])

cols = feats_janela + contexto
dif = pd.DataFrame(casa[cols].to_numpy("float64") - fora[cols].to_numpy("float64"),
                   index=casa.index, columns=[f"dif_{c}" for c in cols])
niveis = pd.DataFrame({
    "casa_rest_days": casa["rest_days"], "fora_rest_days": fora["rest_days"],
    "casa_back_to_back": casa["back_to_back"], "fora_back_to_back": fora["back_to_back"],
}, index=casa.index)

jogos = pd.concat([jogos, dif, niveis], axis=1)
jogos["temporada_anomala"] = jogos["temporada"].isin(TEMPORADAS_ANOMALAS).astype("int8")
jogos = jogos.reset_index().sort_values(["data", "game_id"]).reset_index(drop=True)
print(f"{len(jogos):,} partidas × {len(colunas_features(jogos))} variáveis candidatas")

24,350 partidas × 89 variáveis candidatas


## 6. Aquecimento e descarte por valores ausentes

Primeiro saem as partidas em que alguma das equipes ainda não completou 10 jogos na temporada.
Depois saem as variáveis com mais de 20% de ausentes em alguma temporada. O corte é avaliado
**por temporada**, e não na média geral: uma coluna vazia num único ano parece saudável na
média (5%) e ainda assim inviabilizaria o fold de validação que testa naquele ano. Na base,
a temporada de 2022 vem sem pontos no garrafão, de contra-ataque, de segunda chance e do banco
(seção 7 da etapa 1), e as variáveis derivadas deles caem aqui.

O critério foi avaliado sobre as 20 temporadas de uma vez. Como não envolve a variável
resposta, é tratado como definição prévia de quais campos a base oferece.

In [7]:
antes = len(jogos)
jogos = jogos[jogos["min_jogos_anteriores"] >= MIN_JOGOS_AQUECIMENTO].drop(
    columns="min_jogos_anteriores")
print(f"aquecimento: {antes - len(jogos):,} partidas removidas, {len(jogos):,} restantes")

feats = colunas_features(jogos)
ausentes = jogos.groupby("temporada")[feats].apply(lambda t: t.isna().mean())
pior = ausentes.max()
descartadas = pior[pior > LIMITE_AUSENTES_TEMPORADA].sort_values(ascending=False)
print(f"\n{len(descartadas)} variáveis descartadas por ausentes em alguma temporada:")
for c, v in descartadas.items():
    print(f"   {c}: {v:.0%} ausente em {ausentes[c].idxmax()}")
jogos = jogos.drop(columns=list(descartadas.index))
descartadas = list(descartadas.index)

restantes = jogos[colunas_features(jogos)].isna().sum()
print(f"\nvalores ausentes que restam no conjunto final: {int(restantes.sum())}")

aquecimento: 3,140 partidas removidas, 21,210 restantes

12 variáveis descartadas por ausentes em alguma temporada:
   dif_j10_pct_garrafao: 100% ausente em 2022
   dif_j10_pct_contra_ataque: 100% ausente em 2022
   dif_j10_pct_segunda_chance: 100% ausente em 2022
   dif_j10_pct_banco: 100% ausente em 2022
   dif_j10_opp_pct_garrafao: 100% ausente em 2022
   dif_j10_opp_pct_contra_ataque: 100% ausente em 2022
   dif_tmp_pct_garrafao: 100% ausente em 2022
   dif_tmp_pct_contra_ataque: 100% ausente em 2022
   dif_tmp_pct_segunda_chance: 100% ausente em 2022
   dif_tmp_pct_banco: 100% ausente em 2022
   dif_tmp_opp_pct_garrafao: 100% ausente em 2022
   dif_tmp_opp_pct_contra_ataque: 100% ausente em 2022

valores ausentes que restam no conjunto final: 0


## 7. Conjunto final de variáveis

Uma **família** é a mesma medida observada em janelas diferentes: `dif_j5_ortg`,
`dif_j10_ortg` e `dif_tmp_ortg` formam a família `ortg`. A etapa 4 soma a importância das
colunas de cada família, porque colunas da mesma família são fortemente correlacionadas e o
SHAP divide o crédito entre elas.

In [8]:
feats = colunas_features(jogos)
familia = (pd.Series(feats, index=feats)
           .str.replace(r"^(dif|casa|fora)_", "", regex=True)
           .str.replace(r"^(j\d+|tmp)_", "", regex=True))
tabela = (pd.DataFrame({"familia": familia, "coluna": feats})
          .groupby("familia", sort=False)["coluna"].agg(colunas="size", primeira_coluna="first"))
print(f"{len(feats)} variáveis em {len(tabela)} famílias\n")
print(tabela.to_string())

77 variáveis em 28 famílias

                    colunas             primeira_coluna
familia                                                
ortg                      3                 dif_j5_ortg
drtg                      3                 dif_j5_drtg
efg                       3                  dif_j5_efg
ts                        3                   dif_j5_ts
fg3_pct                   3              dif_j5_fg3_pct
ft_pct                    3               dif_j5_ft_pct
fg3ar                     3                dif_j5_fg3ar
ftr                       3                  dif_j5_ftr
tov_pct                   3              dif_j5_tov_pct
orb_pct                   3              dif_j5_orb_pct
drb_pct                   3              dif_j5_drb_pct
ast_per100                3           dif_j5_ast_per100
stl_per100                3           dif_j5_stl_per100
blk_per100                3           dif_j5_blk_per100
pf_per100                 3            dif_j5_pf_per100
opp_efg            

## 8. Folds da validação Walk-Forward

Em cada era, o fold *k* treina nas temporadas 1 a *k* e testa na temporada *k* + 1. O
treino é cumulativo e o teste é sempre a temporada seguinte às usadas no ajuste. São 9
folds por era e 18 no total.

A sequência segue a **posição** da temporada, não o ano corrido. Dois folds são marcados
para relato à parte na etapa 3: a que testa em 2012 (`teste_anomalo`, temporada reduzida) e a
que treina até 2019 e testa em 2022 (`salto_temporal`, três anos entre treino e teste por
causa da exclusão de 2020 e 2021).

In [9]:
linhas = []
for era, g in jogos.groupby("era"):
    temporadas = sorted(g["temporada"].unique())
    for k in range(1, len(temporadas)):
        treino, teste = temporadas[:k], temporadas[k]
        linhas.append({
            "era": era, "fold": k,
            "treino_de": treino[0], "treino_ate": treino[-1], "teste": teste,
            "n_treino": int(g["temporada"].isin(treino).sum()),
            "n_teste": int((g["temporada"] == teste).sum()),
            "anos_de_salto": int(teste - treino[-1]),
            "salto_temporal": int(teste - treino[-1] > 1),
            "teste_anomalo": int(teste in TEMPORADAS_ANOMALAS),
        })
folds = pd.DataFrame(linhas)
print(folds.to_string(index=False))

     era  fold  treino_de  treino_ate  teste  n_treino  n_teste  anos_de_salto  salto_temporal  teste_anomalo
Classica     1       2005        2005   2006      1069     1070              1               0              0
Classica     2       2005        2006   2007      2139     1071              1               0              0
Classica     3       2005        2007   2008      3210     1072              1               0              0
Classica     4       2005        2008   2009      4282     1074              1               0              0
Classica     5       2005        2009   2010      5356     1071              1               0              0
Classica     6       2005        2010   2011      6427     1074              1               0              0
Classica     7       2005        2011   2012      7501      832              1               0              1
Classica     8       2005        2012   2013      8333     1072              1               0              0
Classica  

## 9. Resumo e salvamento

A taxa de vitória do mandante em cada era é a acurácia do classificador da moda (que prevê
sempre a classe mais frequente, aqui a vitória do mandante): o piso que os modelos da etapa 3
precisam superar.

In [10]:
print(jogos.groupby("era").agg(temporadas=("temporada", "nunique"),
                               partidas=("game_id", "size"),
                               taxa_mandante=("alvo", "mean")).round(3).to_string())

jogos.to_parquet(DIR_DADOS / "features_jogos.parquet", index=False)
folds.to_csv(DIR_DADOS / "folds_walkforward.csv", index=False)
metadados = {
    "executado_em": datetime.now().isoformat(timespec="seconds"),
    "janelas": JANELAS + ["temporada"],
    "min_jogos_aquecimento": MIN_JOGOS_AQUECIMENTO,
    "descanso_maximo": DESCANSO_MAXIMO,
    "temporadas_anomalas": sorted(TEMPORADAS_ANOMALAS),
    "limite_ausentes_temporada": LIMITE_AUSENTES_TEMPORADA,
    "features_descartadas": descartadas,
    "n_jogos": int(len(jogos)),
    "n_features": len(feats),
    "features": feats,
}
(DIR_DADOS / "metadados_features.json").write_text(
    json.dumps(metadados, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"\nsalvo: {len(jogos):,} partidas × {len(feats)} variáveis "
      f"-> {DIR_DADOS / 'features_jogos.parquet'}")

          temporadas  partidas  taxa_mandante
era                                          
Classica          10     10479          0.599
Moderna           10     10731          0.567



salvo: 21,210 partidas × 77 variáveis -> dados\processados\features_jogos.parquet
